In [38]:
from src.post_processing import PathWrangler
import polars as pl
from pathlib import Path
from collections import defaultdict

In [39]:
study = "/home/stef/quest_data/bottle/data/processed/test"
known = "/home/stef/bottle/artifacts/known"
out_dir = "/home/stef/bottle/artifacts/coa_mutase_paths"

pw = PathWrangler(Path(study), Path(known))

In [6]:
tables = pw.get_paths()

In [7]:
paths = tables['paths']
krs = tables['known_reactions']
prs = tables['predicted_reactions']

pr2krs = dict(zip(prs['id'], prs['analogue_ids'].to_list()))
krs2enzymes = dict(zip(krs['id'], krs['enzymes'].to_list()))
prs2enzymes = defaultdict(list)
for p, ks in pr2krs.items():
    for k in ks:
        prs2enzymes[p] += krs2enzymes[k]

prs2smarts = dict(zip(prs['id'], prs['smarts'].to_list()))

paths = paths.select(
    pl.col("id"),
    pl.col("rxn_id"),
    (pl.col("generation") + 1).alias("step"),
    pl.col("starters").map_elements(lambda x : ";".join(x), return_dtype=pl.String),
    pl.col("targets").map_elements(lambda x : ";".join(x), return_dtype=pl.String),
    pl.col("rxn_id").replace_strict(prs2enzymes, default=[]).map_elements(lambda x : ";".join(x), return_dtype=pl.String).alias("enzymes"),
    pl.col("rxn_id").replace_strict(prs2smarts, default=None).alias("smarts")
)
paths

id,rxn_id,step,starters,targets,enzymes,smarts
str,str,i32,str,str,str,str
"""0a24a1c33085a4f1ea7f8e19ce06c3…","""5582845919356f447668dd0ea4647f…",1,"""pyruvate""","""3hpa""","""A9WGE2;A9WC35;Q8N0X4;S5N020;Q8…","""CC(=O)SCCNC(=O)CCNC(=O)C(O)C(C…"
"""0a24a1c33085a4f1ea7f8e19ce06c3…","""8c16137b4045f00069ba91d38a3b8b…",2,"""pyruvate""","""3hpa""","""A9WC41;Q9JLZ3;Q13825;Q1ZXF1;F1…","""CC(O)(CC(=O)SCCNC(=O)CCNC(=O)C…"
"""0a24a1c33085a4f1ea7f8e19ce06c3…","""9c2c777c0fea421a26efc26857522f…",3,"""pyruvate""","""3hpa""","""Q8VCH6;Q15392;Q5BQE6;Q60HC5;Q3…","""CC(=CC(=O)SCCNC(=O)CCNC(=O)C(O…"
"""0a24a1c33085a4f1ea7f8e19ce06c3…","""e86048734a2f69f1257bd92b62ddb4…",4,"""pyruvate""","""3hpa""","""Q5KUG0;A7IQE6;Q1LRY0;A7IQE5""","""CC(CC(=O)SCCNC(=O)CCNC(=O)C(O)…"
"""0a24a1c33085a4f1ea7f8e19ce06c3…","""bd616dde067e2786bec3a4ea2c6e17…",5,"""pyruvate""","""3hpa""","""Q0P5J1;Q96K12;Q7TNT2;Q0P5J1;Q9…","""NC(=O)C1=CN(C2OC(COP(=O)(O)OP(…"
"""9b13db2d9c8b6a2b40034ef19b269b…","""27450e93bf1eb2d982a541e9b34724…",1,"""glutamic_acid""","""3hpa""","""Q24SG9;Q8RHY7;P80078;Q73KI2;Q5…","""NC(CCC(=O)O)C(=O)O>>CC(C(=O)O)…"
"""9b13db2d9c8b6a2b40034ef19b269b…","""646032df080e79e8522e14d05d8548…",2,"""glutamic_acid""","""3hpa""","""A9X6P9;P76518;B0MC58;G2SYC0;O0…","""CC(C(=O)O)C(N)C(=O)O.CC(=O)SCC…"
"""9b13db2d9c8b6a2b40034ef19b269b…","""62277981d883290bf5c0d4ed258555…",3,"""glutamic_acid""","""3hpa""","""""","""NC(=O)C1=CN(C2OC(COP(=O)(O)OP(…"
"""9b13db2d9c8b6a2b40034ef19b269b…","""e86048734a2f69f1257bd92b62ddb4…",4,"""glutamic_acid""","""3hpa""","""Q5KUG0;A7IQE6;Q1LRY0;A7IQE5""","""CC(CC(=O)SCCNC(=O)CCNC(=O)C(O)…"


Direct look at parquet files

In [42]:
paths = pl.read_parquet(
    Path(study) / "paths.parquet"
)
paths.head()

path_id,rxn_id,main_pdt_id,rxn_type,generation
str,str,str,enum,i32
"""6a00d985f648ce51059730c60da836…","""48960619c0f4a41ba252393d7e1b84…","""31e1170961d93fe3e7baae8ae55110…","""predicted""",0
"""6a00d985f648ce51059730c60da836…","""3238d364a7911ad3440c41c05c944a…","""cd431e4c6b054540791647925df16a…","""predicted""",1
"""6a00d985f648ce51059730c60da836…","""e00d2fd6f101dd3d9ac4cda139cf0c…","""7adf4b7cf3272a4091055af7ffa13e…","""predicted""",2
"""6a00d985f648ce51059730c60da836…","""bd616dde067e2786bec3a4ea2c6e17…","""6569ea8ebe9fdd8627adb76f7d942d…","""predicted""",4
"""6a00d985f648ce51059730c60da836…","""e86048734a2f69f1257bd92b62ddb4…","""96d1eae18546609d0837661cc2b156…","""predicted""",3


In [43]:
path_stats = pl.read_parquet(
    Path(study) / "path_stats.parquet"
)
path_stats.head()

id,starters,targets,dg_opt,dg_err,starter_ids,target_ids,mdf,mean_max_rxn_sim,mean_mean_rxn_sim,min_max_rxn_sim,min_mean_rxn_sim,feasibility_frac
str,list[str],list[str],list[f32],list[f32],list[str],list[str],f32,f32,f32,f32,f32,f32
"""6a00d985f648ce51059730c60da836…","[""pyruvate"", ""methionine""]","[""3hpa""]",null,null,"[""2efe03c1b1595e5fedce6a3674dad82075142215"", ""1b71170a51728c7cbd17c0d40b1f576adbdc408b""]","[""6569ea8ebe9fdd8627adb76f7d942dbbbb83c56d""]",null,null,null,null,null,null


In [24]:
ps = path_stats.filter(
    pl.col("id") == '6a00d985f648ce51059730c60da836938782ae29'
).unique()

In [25]:
for elt in ps.group_by("id"):
    print(elt)

(('6a00d985f648ce51059730c60da836938782ae29',), shape: (2, 13)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ id        ┆ starters  ┆ targets   ┆ dg_opt    ┆ … ┆ mean_mean ┆ min_max_r ┆ min_mean_ ┆ feasibil │
│ ---       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ _rxn_sim  ┆ xn_sim    ┆ rxn_sim   ┆ ity_frac │
│ str       ┆ list[str] ┆ list[str] ┆ list[f32] ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│           ┆           ┆           ┆           ┆   ┆ f32       ┆ f32       ┆ f32       ┆ f32      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 6a00d985f ┆ ["methion ┆ ["3hpa"]  ┆ [-18.1495 ┆ … ┆ 0.641469  ┆ 0.454545  ┆ 0.409714  ┆ 1.0      │
│ 648ce5105 ┆ ine"]     ┆           ┆ 93, -18.1 ┆   ┆           ┆           ┆           ┆          │
│ 9730c60da ┆           ┆           ┆ 49593, …  ┆   ┆           ┆           ┆           ┆          │
│ 836…      ┆           ┆   

In [ ]:
ps.group_by("id").agg(
    pl.col("starters").flatten().unique().alias("starters"),
    

)

id,starters
str,list[str]
"""6a00d985f648ce51059730c60da836…","[""methionine"", ""pyruvate""]"


In [28]:
ps.explode(['starters', 'targets', 'starter_ids', 'target_ids']).group_by('id').agg()

id
str
"""6a00d985f648ce51059730c60da836…"


In [22]:
paths.filter(
    pl.col("path_id") == '6a00d985f648ce51059730c60da836938782ae29'
).unique()

path_id,rxn_id,main_pdt_id,rxn_type,generation
str,str,str,enum,i32
"""6a00d985f648ce51059730c60da836…","""3238d364a7911ad3440c41c05c944a…","""cd431e4c6b054540791647925df16a…","""predicted""",1
"""6a00d985f648ce51059730c60da836…","""48960619c0f4a41ba252393d7e1b84…","""31e1170961d93fe3e7baae8ae55110…","""predicted""",0
"""6a00d985f648ce51059730c60da836…","""bd616dde067e2786bec3a4ea2c6e17…","""6569ea8ebe9fdd8627adb76f7d942d…","""predicted""",4
"""6a00d985f648ce51059730c60da836…","""e86048734a2f69f1257bd92b62ddb4…","""96d1eae18546609d0837661cc2b156…","""predicted""",3
"""6a00d985f648ce51059730c60da836…","""e00d2fd6f101dd3d9ac4cda139cf0c…","""7adf4b7cf3272a4091055af7ffa13e…","""predicted""",2


In [7]:
whitelist = paths.filter(
    pl.col("path_id") == '6a00d985f648ce51059730c60da836938782ae29'
)["rxn_id"].unique().to_list()

In [8]:
am_rxns = pl.read_parquet(
    "/home/stef/quest_data/bottle/data/interim/expansions/5_steps_ccm_aa_to_bottle_targets_25_combo_mechinferred_dt_13_and_mechinferred_dt_04_rules/am_rxns.parquet"
)
am_rxns.head()

smarts,am_smarts,rules,half_expansion,size
str,str,list[str],enum,i32
"""CC(C)(C(=O)OP(=O)(O)O)C(O)C(OO…","""[CH3:1][C:2]([CH3:3])([C:4](=[…","[""mechinferred_dt_04:1759""]","""retro""",7
"""CC(=O)OP(=O)(O)OC1C(COP(=O)(O)…","""[CH3:28][C:29](=[O:30])[O:31][…","[""mechinferred_dt_04:6200"", ""mechinferred_dt_04:627"", ""mechinferred_dt_04:1547""]","""retro""",17
"""NC(=O)C1=CN(C2OC(COP(=O)(O)OP(…","""[NH2:92][C:93](=[O:94])[C:95]1…","[""mechinferred_dt_04:1439""]","""retro""",12
"""CC(=O)Nc1ncnc2c1ncn2C1OC(COP(=…","""[CH3:58][C:59](=[O:60])[NH:61]…","[""mechinferred_dt_04:1069""]","""retro""",5
"""O=c1ccn(C2OC(COP(=O)(O)OP(=O)(…","""[O:1]=[c:2]1[cH:3][cH:4][n:5](…","[""mechinferred_dt_04:9057""]","""retro""",20


In [32]:
pred_rxns = pl.read_parquet(
    Path(study) / "predicted_reactions.parquet"
)
pred_rxns.filter(
    pl.col("id").str.starts_with("4896061")
)["smarts"][0]

'CC(=O)C(=O)O.NC(C=O)CC(=O)O.CSCCC(N)C(=O)O>>NC(CO)CC(=O)O.C[S+](CCC(N)C(=O)O)CC(=O)C(=O)O'

In [ ]:

pred_rxns = pred_rxns.filter(
    pl.col("id").is_in(whitelist)
).select(
    ['smarts', 'am_smarts', 'rules']
).with_columns(
    pl.lit(8).alias("size")
)
pred_rxns.head()

In [15]:
def get_dir(rules: list[str]) -> str:
    rule_set = list(set(
        [r.split(":")[0] for r in rules]
    ))[0]
    if rule_set.startswith("mechinferred_dt_13"):
        return "forward"
    elif rule_set.startswith("mechinferred_dt_04"):
        return "retro"

In [16]:
pred_rxns = pred_rxns.with_columns(
    pl.col("rules").map_elements(get_dir, return_dtype=pl.String).alias("half_expansion")
)
pred_rxns.head()

smarts,am_smarts,rules,size,half_expansion
str,str,list[str],i32,str
"""CC(=O)C(=O)O.NC(C=O)CC(=O)O.CS…","""[CH3:1][C:2](=[O:3])[C:4](=[O:…","[""mechinferred_dt_13:681"", ""mechinferred_dt_13:681"", ""mechinferred_dt_13:681""]",8,"""forward"""
"""NC(=O)C1=CN(C2OC(COP(=O)(O)OP(…","""[NH2:105][C:106](=[O:107])[C:1…","[""mechinferred_dt_04:4272"", ""mechinferred_dt_04:1361""]",8,"""retro"""
"""C[S+](CCC(N)C(=O)O)CC(=O)C(=O)…","""[CH3:1][S+:2]([CH2:3][CH2:4][C…","[""mechinferred_dt_13:1623"", ""mechinferred_dt_13:1623""]",8,"""forward"""
"""CC(CC(=O)SCCNC(=O)CCNC(=O)C(O)…","""[CH3:3][CH:2]([CH2:1][C:7](=[O…","[""mechinferred_dt_04:3939""]",8,"""retro"""
"""CC(CC(=O)O)C(=O)O.Nc1ncnc2c1nc…","""[CH3:1][CH:2]([CH2:3][C:4](=[O…","[""mechinferred_dt_04:1281"", ""mechinferred_dt_04:526""]",8,"""retro"""


In [21]:
path_stats.unique()

id,starters,targets,dg_opt,dg_err,starter_ids,target_ids,mdf,mean_max_rxn_sim,mean_mean_rxn_sim,min_max_rxn_sim,min_mean_rxn_sim,feasibility_frac
str,list[str],list[str],list[f32],list[f32],list[str],list[str],f32,f32,f32,f32,f32,f32
"""e609b2cc97b0b40f3e6f9ac402fbdd…","[""serine""]","[""3hpa""]","[170.994858, 47.267452, … -868.033936]","[-0.000008, 4.4223e-7, … 100000.0]","[""26d1da597f251b7677489fe81ff93b9675c82fb2""]","[""6569ea8ebe9fdd8627adb76f7d942dbbbb83c56d""]",-170.994858,0.687657,0.65158,0.307793,0.307793,0.75
"""2d12cf0fe313bcfa031e42efcf833b…","[""pyruvate""]","[""3hpa""]","[-348.431061, -348.431061, … -34.956501]","[-141421.359375, -141421.359375, … -6.5983e-7]","[""2efe03c1b1595e5fedce6a3674dad82075142215""]","[""6569ea8ebe9fdd8627adb76f7d942dbbbb83c56d""]",19.884821,0.658988,0.633617,0.454545,0.409714,1.0
"""0bd539b8dc9cca75345e227839694c…","[""tryptophan""]","[""3hpa""]","[-20.643488, -20.514862, … -26.510054]","[0.000004, -0.000005, … 8.7741e-8]","[""da568041b75aa8f08b7b7a33c8f6eee3c4bde0d1""]","[""6569ea8ebe9fdd8627adb76f7d942dbbbb83c56d""]",19.884821,0.772447,0.700924,0.681818,0.537792,1.0
"""1f0df8349b744537e10684002e116f…","[""aspartic_acid""]","[""3hpa""]","[-11.956059, -18.818769, … -22.641579]","[0.000009, 100000.0, … 8.7741e-8]","[""6498dc15cfaf4e66aa91145abc0c0cc648898075""]","[""6569ea8ebe9fdd8627adb76f7d942dbbbb83c56d""]",11.956059,0.72572,0.705974,0.636364,0.609925,0.6
"""05ab14379b6a8d1d6253be6e326cc1…","[""oxaloacetate""]","[""3hpa""]","[-7.620832, -7.620832, … -12.388758]","[-100000.0, -100000.0, … 4.1173e-7]","[""20e696beb1325772afccb7eb1d2c187c70696753""]","[""6569ea8ebe9fdd8627adb76f7d942dbbbb83c56d""]",4.892096,0.679686,0.640281,0.454545,0.409714,0.8
…,…,…,…,…,…,…,…,…,…,…,…,…
"""3f01e4e5678e7e1ce0f5f372b81d1a…","[""ketoglutarate""]","[""3hpa""]","[-26.905195, -22.492613, … -27.320425]","[0.000003, 0.000003, … 8.7741e-8]","[""6cf6db1d70f43061de31deeb417214c6fd7ad005""]","[""6569ea8ebe9fdd8627adb76f7d942dbbbb83c56d""]",19.884821,0.752675,0.738933,0.681818,0.627544,1.0
"""1cbcc4bc6059cd23ad0e059a6ce68c…","[""methionine""]","[""3hpa""]","[-780.289734, -780.289734, … -15.687915]","[-141421.359375, -141421.359375, … -9.1728e-7]","[""1b71170a51728c7cbd17c0d40b1f576adbdc408b""]","[""6569ea8ebe9fdd8627adb76f7d942dbbbb83c56d""]",-854.374146,0.631261,0.616918,0.454545,0.409714,1.0
"""2e0ef32b970982a52010969c2b8b73…","[""malate""]","[""3hpa""]","[854.368958, 31.707809, … -7.988899]","[0.000004, -0.000005, … 8.7741e-8]","[""77d21e11cc906a4c9614c97db30147761a24e690""]","[""6569ea8ebe9fdd8627adb76f7d942dbbbb83c56d""]",-854.368958,0.774561,0.703975,0.681818,0.555789,1.0


In [17]:
pred_rxns.write_parquet(
    "/home/stef/quest_data/bottle/data/interim/expansions/test/am_rxns.parquet")

In [34]:
cpds = pl.read_parquet(
    "/home/stef/quest_data/bottle/data/interim/expansions/test/compounds.parquet"
)
cpds.head()

id,smiles,type,name
str,str,enum,str
"""fae11404b12230ded37cfd2511bf3d…","""O=C(O)CCC(=O)O""","""source""","""succinate"""
"""55e7ca03fb953e8974749691680048…","""O=C(O)C=CC(=O)O""","""source""","""fumarate"""
"""77d21e11cc906a4c9614c97db30147…","""O=C(O)CC(O)C(=O)O""","""source""","""malate"""
"""20e696beb1325772afccb7eb1d2c18…","""O=C(O)CC(=O)C(=O)O""","""source""","""oxaloacetate"""
"""6cf6db1d70f43061de31deeb417214…","""O=C(O)CCC(=O)C(=O)O""","""source""","""ketoglutarate"""


In [35]:
cpds.filter(
    pl.col("smiles") == "NC(C=O)CC(=O)O")

id,smiles,type,name
str,str,enum,str


Formatting a pull for collaborators

In [ ]:
enzymes = tables['enzymes']


enzymes.with_columns(
    pl.col("id").map_elements(lambda x : f"https://www.uniprot.org/uniprotkb/{x}/entry", return_dtype=pl.String).alias("link"),
)

In [ ]:
paths.write_csv(
    Path(out_dir) / "250912_3hpa_paths.csv",
    separator=','
)

enzymes.write_csv(
    Path(out_dir) / "250912_3hpa_enzymes.csv",
    separator=','
)

In [ ]:
from ergochemics.draw import draw_reaction
from IPython.display import SVG, display

In [ ]:
paths.filter(
    pl.col("id").str.starts_with("0a24a1c")
)

In [ ]:
for row in paths.filter(pl.col("id").str.starts_with("0a24a1c")).iter_rows(named=True):
    smarts = row['smarts']
    print(row['rxn_id'])
    display(SVG(draw_reaction(smarts)))

In [ ]:
prs.filter(
    pl.col("id") == "9c2c777c0fea421a26efc26857522fd2ccce1555"
)

In [ ]:
min_maps = pl.read_parquet(
    "/home/stef/bottle/artifacts/rxn_x_rule_mapping/mapped_known_reactions_x_rc_plus_0_rules.parquet"
)
min_maps.filter(
    pl.col("rxn_id") == "d7eea5a1a938257639a2cbe5950fa3501a3fc821"
)

In [ ]:
row['rxn_id']


In [ ]:
for row in min_maps.filter(pl.col("rule_id") == 515).iter_rows(named=True):
    smarts = row['smarts']
    print(row['rxn_id'])
    print(krs.filter(pl.col("id") == row['rxn_id'])['db_ids'])
    display(SVG(draw_reaction(smarts)))